# Conferência DEC/FEC — ADMStoIQS

Objetivo: conferir por que o DEC/FEC mudou após os ajustes de apuração dos indicadores.

Este notebook compara:

- denominador por `COUNT(DISTINCT NUM_UC_UCI)`;
- denominador IQS `mart_consumidor_faturado_regional_[anomes].parquet`;
- efeito dos filtros líquidos;
- valores materializados em `indicadores_comparativo_[anomes].parquet`;
- cálculo direto a partir da apuração OMS antes/depois.

Use para responder perguntas como: “por que o DEC antes saiu de 0,762 para 1?”.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

ROOT = Path(r"D:\ADMStoIQS")
ANOMES = "202605"

apuracao_antes = ROOT / "data" / "mart" / "apuracao" / f"agrupamento_oms_APURACAO_{ANOMES}.parquet"
apuracao_depois = ROOT / "data" / "mart" / "apuracao" / f"agrupamento_oms_APURACAO_{ANOMES}_TRATADO.parquet"

indicadores_uc = ROOT / "data" / "mart" / "indicadores" / f"indicadores_uc_{ANOMES}.parquet"
indicadores_agregado = ROOT / "data" / "mart" / "indicadores" / f"indicadores_agregado_{ANOMES}.parquet"
indicadores_comparativo = ROOT / "data" / "mart" / "indicadores" / f"indicadores_comparativo_{ANOMES}.parquet"

den_iqs_mart = ROOT / "data" / "external" / "iqs" / "mart" / f"mart_consumidor_faturado_regional_{ANOMES}.parquet"
den_iqs_raw = ROOT / "data" / "external" / "iqs" / "raw" / f"consumidor_faturado_regional_{ANOMES}.parquet"
uc_faturada_raw = ROOT / "data" / "external" / "iqs" / "raw" / f"uc_faturada_hcai_{ANOMES}.parquet"

paths = {
    "apuracao_antes": apuracao_antes,
    "apuracao_depois": apuracao_depois,
    "indicadores_uc": indicadores_uc,
    "indicadores_agregado": indicadores_agregado,
    "indicadores_comparativo": indicadores_comparativo,
    "den_iqs_mart": den_iqs_mart,
    "den_iqs_raw": den_iqs_raw,
    "uc_faturada_raw": uc_faturada_raw,
}

pd.DataFrame([{"arquivo": k, "existe": v.exists(), "caminho": str(v)} for k, v in paths.items()])

## 1. Conferir valores já materializados

Primeiro, veja o que está gravado no mart oficial de indicadores.

In [ ]:
con = duckdb.connect(database=":memory:")

def read_parquet_df(path: Path, sql: str = "SELECT * FROM read_parquet(?)"):
    if not path.exists():
        return pd.DataFrame()
    return con.execute(sql, [str(path)]).df()

comparativo_df = read_parquet_df(
    indicadores_comparativo,
    """
    SELECT *
    FROM read_parquet(?)
    WHERE nivel IN ('COPEL', 'REGIONAL')
    ORDER BY nivel, regional_origem
    """,
)
comparativo_df

## 2. Conferir denominadores

O DEC/FEC muda diretamente quando muda o divisor.

Fórmulas:

```text
DEC = soma(DIC horas) / quantidade_ucs
FEC = soma(FIC) / quantidade_ucs
```

In [ ]:
def describe_columns(path: Path):
    if not path.exists():
        return pd.DataFrame()
    return con.execute("DESCRIBE SELECT * FROM read_parquet(?)", [str(path)]).df()

print("Denominador IQS mart:", den_iqs_mart)
display(describe_columns(den_iqs_mart if den_iqs_mart.exists() else den_iqs_raw))

In [ ]:
den_path = den_iqs_mart if den_iqs_mart.exists() else den_iqs_raw

if den_path.exists():
    cols = [r[0] for r in con.execute("DESCRIBE SELECT * FROM read_parquet(?)", [str(den_path)]).fetchall()]
    print(cols)
    display(con.execute("SELECT * FROM read_parquet(?) LIMIT 20", [str(den_path)]).df())
else:
    print("Denominador IQS não encontrado.")

## 3. Montar base de auditoria OMS

Esta etapa replica a lógica do materializador de indicadores, mas permite ligar/desligar filtros para medir impacto.

In [ ]:
con.execute("DROP TABLE IF EXISTS base_raw")
con.execute("DROP TABLE IF EXISTS base_auditoria")

con.execute(
    """
    CREATE TEMP TABLE base_raw AS
    SELECT 'antes' AS cenario, * FROM read_parquet(?)
    UNION ALL BY NAME
    SELECT 'depois' AS cenario, * FROM read_parquet(?)
    """,
    [str(apuracao_antes), str(apuracao_depois)],
)

con.execute(
    """
    CREATE TEMP TABLE base_auditoria AS
    WITH parse AS (
        SELECT
            cenario,
            CAST(NUM_UC_UCI AS VARCHAR) AS num_uc_uci,
            CAST(NUM_POSTO_UCI AS VARCHAR) AS num_posto_uci,
            CAST(NUM_SEQ_INTRP AS VARCHAR) AS num_seq_intrp,
            CAST(NUM_INTRP_UCI AS VARCHAR) AS num_intrp_uci,
            CAST(ESTADO_INTRP AS VARCHAR) AS estado_intrp,
            CAST(NUM_MOTIVO_TRAT_DIF_UCI AS VARCHAR) AS motivo_tratamento,
            CAST(TIPO_PROTOC_JUSTIF_UCI AS VARCHAR) AS tipo_protocolo_uci,
            CAST(TIPO_PROTOC_JUSTIF_INTRP AS VARCHAR) AS tipo_protocolo_intrp,
            CASE
                WHEN UPPER(TRIM(COALESCE(NULLIF(SIGLA_REGIONAL, ''), NULLIF(REGIONAL_ORIGEM, '')))) = 'P' THEN 'CSL'
                WHEN UPPER(TRIM(COALESCE(NULLIF(SIGLA_REGIONAL, ''), NULLIF(REGIONAL_ORIGEM, '')))) = 'L' THEN 'NRT'
                WHEN UPPER(TRIM(COALESCE(NULLIF(SIGLA_REGIONAL, ''), NULLIF(REGIONAL_ORIGEM, '')))) = 'M' THEN 'NRO'
                WHEN UPPER(TRIM(COALESCE(NULLIF(SIGLA_REGIONAL, ''), NULLIF(REGIONAL_ORIGEM, '')))) = 'C' THEN 'LES'
                WHEN UPPER(TRIM(COALESCE(NULLIF(SIGLA_REGIONAL, ''), NULLIF(REGIONAL_ORIGEM, '')))) = 'V' THEN 'OES'
                WHEN UPPER(TRIM(COALESCE(NULLIF(SIGLA_REGIONAL, ''), NULLIF(REGIONAL_ORIGEM, '')))) IN ('CSL', 'NRT', 'NRO', 'LES', 'OES')
                    THEN UPPER(TRIM(COALESCE(NULLIF(SIGLA_REGIONAL, ''), NULLIF(REGIONAL_ORIGEM, ''))))
                ELSE 'COPEL'
            END AS regional_origem,
            COALESCE(
                try_strptime(NULLIF(DTHR_INICIO_INTRP_UC, ''), '%Y-%m-%d %H:%M:%S'),
                try_strptime(NULLIF(DTHR_INICIO_INTRP_UC, ''), '%d/%m/%Y %H:%M:%S'),
                try_strptime(NULLIF(DATA_HORA_INIC_INTRP, ''), '%Y-%m-%d %H:%M:%S'),
                try_strptime(NULLIF(DATA_HORA_INIC_INTRP, ''), '%d/%m/%Y %H:%M:%S')
            ) AS inicio_uc,
            COALESCE(
                try_strptime(NULLIF(DATA_HORA_FIM_INTRP, ''), '%Y-%m-%d %H:%M:%S'),
                try_strptime(NULLIF(DATA_HORA_FIM_INTRP, ''), '%d/%m/%Y %H:%M:%S')
            ) AS fim_uc
        FROM base_raw
        WHERE NUM_UC_UCI IS NOT NULL
          AND NULLIF(CAST(NUM_UC_UCI AS VARCHAR), '') IS NOT NULL
    )
    SELECT
        *,
        date_diff('minute', inicio_uc, fim_uc) AS duracao_minutos_uc,
        inicio_uc IS NOT NULL AND fim_uc IS NOT NULL AND date_diff('minute', inicio_uc, fim_uc) >= 0 AS filtro_datas_validas,
        date_diff('minute', inicio_uc, fim_uc) >= 3 AS filtro_duracao_3min,
        estado_intrp = '4' AS filtro_estado_4,
        NULLIF(TRIM(tipo_protocolo_uci), '') = '0' AS filtro_protocolo_uci_0,
        motivo_tratamento IS NULL OR NULLIF(TRIM(motivo_tratamento), '') IS NULL AS filtro_motivo_nulo
    FROM parse
    """
)

con.execute("SELECT COUNT(*) AS linhas FROM base_auditoria").df()

## 4. Medir impacto de cada filtro

Esta tabela mostra quantas linhas sobram após cada etapa. Se algum filtro derrubar ou aumentar comportamento inesperado, o problema aparece aqui.

In [ ]:
impacto_filtros = con.execute(
    """
    SELECT
        cenario,
        COUNT(*) AS total_linhas,
        COUNT(DISTINCT num_uc_uci) AS ucs_distintas_total,
        SUM(CASE WHEN filtro_datas_validas THEN 1 ELSE 0 END) AS linhas_datas_validas,
        SUM(CASE WHEN filtro_datas_validas AND filtro_duracao_3min THEN 1 ELSE 0 END) AS linhas_3min,
        SUM(CASE WHEN filtro_datas_validas AND filtro_duracao_3min AND filtro_estado_4 THEN 1 ELSE 0 END) AS linhas_estado_4,
        SUM(CASE WHEN filtro_datas_validas AND filtro_duracao_3min AND filtro_estado_4 AND filtro_protocolo_uci_0 THEN 1 ELSE 0 END) AS linhas_protocolo_0,
        SUM(CASE WHEN filtro_datas_validas AND filtro_duracao_3min AND filtro_estado_4 AND filtro_protocolo_uci_0 AND filtro_motivo_nulo THEN 1 ELSE 0 END) AS linhas_liquidas_sem_faturamento,
        COUNT(DISTINCT CASE WHEN filtro_datas_validas AND filtro_duracao_3min AND filtro_estado_4 AND filtro_protocolo_uci_0 AND filtro_motivo_nulo THEN num_uc_uci END) AS ucs_liquidas_sem_faturamento
    FROM base_auditoria
    GROUP BY cenario
    ORDER BY cenario
    """
).df()
impacto_filtros

## 5. Aplicar UC faturada HCAI

Aqui entra o filtro `faturado = 'S'` vindo do HCAI/IQS.

In [ ]:
con.execute("DROP TABLE IF EXISTS ucs_faturadas_hcai")

if uc_faturada_raw.exists():
    con.execute(
        """
        CREATE TEMP TABLE ucs_faturadas_hcai AS
        SELECT DISTINCT
            CAST(uc AS VARCHAR) AS num_uc_uci,
            UPPER(TRIM(CAST(faturado AS VARCHAR))) AS faturado,
            CAST(regional AS VARCHAR) AS regional_hcai
        FROM read_parquet(?)
        """,
        [str(uc_faturada_raw)],
    )
    print(con.execute("SELECT COUNT(*) FROM ucs_faturadas_hcai").fetchone()[0])
else:
    print("UC faturada HCAI não encontrada:", uc_faturada_raw)

In [ ]:
if uc_faturada_raw.exists():
    impacto_faturamento = con.execute(
        """
        SELECT
            base.cenario,
            COUNT(*) AS linhas_liquidas_faturadas,
            COUNT(DISTINCT base.num_uc_uci) AS ucs_liquidas_faturadas,
            SUM(base.duracao_minutos_uc) / 60.0 AS soma_dic_horas,
            COUNT(DISTINCT base.num_seq_intrp) AS soma_fic
        FROM base_auditoria base
        INNER JOIN ucs_faturadas_hcai fat
          ON base.num_uc_uci = fat.num_uc_uci
         AND fat.faturado = 'S'
        WHERE base.filtro_datas_validas
          AND base.filtro_duracao_3min
          AND base.filtro_estado_4
          AND base.filtro_protocolo_uci_0
          AND base.filtro_motivo_nulo
        GROUP BY base.cenario
        ORDER BY base.cenario
        """
    ).df()
    display(impacto_faturamento)
else:
    print("Sem filtro de faturamento aplicado.")

## 6. Comparar DEC/FEC por tipo de denominador

Aqui está a conferência principal. O mesmo numerador é dividido por dois denominadores:

- `denominador_oms`: UCs distintas da base líquida;
- `denominador_iqs`: fonte oficial materializada do IQS, quando disponível.

In [ ]:
def detectar_colunas_denominador(path: Path):
    cols = [r[0] for r in con.execute("DESCRIBE SELECT * FROM read_parquet(?)", [str(path)]).fetchall()]
    up = {c.upper(): c for c in cols}
    regional = next((up[c] for c in ["REGIONAL_TOTAL", "REGIONAL", "SIGLA_REGIONAL", "DSC_REGIONAL"] if c in up), None)
    qtd = next((up[c] for c in ["UC_FATURADA", "QTD_UC", "QTD_UCS", "QUANTIDADE_UCS", "CONSUMIDORES", "TOTAL", "QTDE"] if c in up), None)
    return regional, qtd, cols

if den_path.exists():
    regional_col, qtd_col, cols = detectar_colunas_denominador(den_path)
    print("regional_col=", regional_col, "qtd_col=", qtd_col)
    print(cols)
    if regional_col and qtd_col:
        con.execute("DROP TABLE IF EXISTS denominador_iqs")
        con.execute(
            f"""
            CREATE TEMP TABLE denominador_iqs AS
            SELECT
                COALESCE(NULLIF(CAST({regional_col} AS VARCHAR), ''), 'SEM_REGIONAL') AS regional_origem,
                SUM(TRY_CAST({qtd_col} AS DOUBLE)) AS quantidade_ucs_iqs
            FROM read_parquet(?)
            GROUP BY COALESCE(NULLIF(CAST({regional_col} AS VARCHAR), ''), 'SEM_REGIONAL')
            """,
            [str(den_path)],
        )
        display(con.execute("SELECT * FROM denominador_iqs ORDER BY regional_origem").df())
else:
    print("Denominador IQS indisponível.")

In [ ]:
if uc_faturada_raw.exists():
    con.execute("DROP TABLE IF EXISTS numerador_liquido")
    con.execute(
        """
        CREATE TEMP TABLE numerador_liquido AS
        SELECT
            base.cenario,
            base.regional_origem,
            base.num_uc_uci,
            SUM(base.duracao_minutos_uc) / 60.0 AS dic_horas,
            COUNT(DISTINCT base.num_seq_intrp) AS fic,
            MAX(base.duracao_minutos_uc) / 60.0 AS dmic_horas
        FROM base_auditoria base
        INNER JOIN ucs_faturadas_hcai fat
          ON base.num_uc_uci = fat.num_uc_uci
         AND fat.faturado = 'S'
        WHERE base.filtro_datas_validas
          AND base.filtro_duracao_3min
          AND base.filtro_estado_4
          AND base.filtro_protocolo_uci_0
          AND base.filtro_motivo_nulo
        GROUP BY base.cenario, base.regional_origem, base.num_uc_uci
        """
    )

    comparacao_denominadores = con.execute(
        """
        WITH reg AS (
            SELECT
                cenario,
                regional_origem,
                COUNT(DISTINCT num_uc_uci) AS denominador_oms,
                SUM(dic_horas) AS soma_dic_horas,
                SUM(fic) AS soma_fic
            FROM numerador_liquido
            GROUP BY cenario, regional_origem
        ),
        copel AS (
            SELECT
                cenario,
                'COPEL' AS regional_origem,
                SUM(denominador_oms) AS denominador_oms,
                SUM(soma_dic_horas) AS soma_dic_horas,
                SUM(soma_fic) AS soma_fic
            FROM reg
            GROUP BY cenario
        ),
        todos AS (
            SELECT * FROM reg
            UNION ALL
            SELECT * FROM copel
        ),
        den_iqs AS (
            SELECT regional_origem, quantidade_ucs_iqs FROM denominador_iqs
            UNION ALL
            SELECT 'COPEL', SUM(quantidade_ucs_iqs) FROM denominador_iqs
        )
        SELECT
            todos.cenario,
            todos.regional_origem,
            todos.soma_dic_horas,
            todos.soma_fic,
            todos.denominador_oms,
            den_iqs.quantidade_ucs_iqs AS denominador_iqs,
            todos.soma_dic_horas / NULLIF(todos.denominador_oms, 0) AS dec_com_den_oms,
            todos.soma_dic_horas / NULLIF(den_iqs.quantidade_ucs_iqs, 0) AS dec_com_den_iqs,
            todos.soma_fic / NULLIF(todos.denominador_oms, 0) AS fec_com_den_oms,
            todos.soma_fic / NULLIF(den_iqs.quantidade_ucs_iqs, 0) AS fec_com_den_iqs
        FROM todos
        LEFT JOIN den_iqs ON den_iqs.regional_origem = todos.regional_origem
        ORDER BY todos.regional_origem, todos.cenario
        """
    ).df()
    display(comparacao_denominadores)
else:
    print("Não foi possível calcular: UC faturada HCAI indisponível.")

## 7. Conferir COPEL antes/depois

Use esta célula para bater especificamente o valor que aparece no card do gestor.

In [ ]:
if 'comparacao_denominadores' in globals():
    display(comparacao_denominadores[comparacao_denominadores['regional_origem'].eq('COPEL')])

if not comparativo_df.empty:
    display(comparativo_df[comparativo_df['nivel'].eq('COPEL')])

## 8. Diagnóstico rápido

Se `dec_com_den_oms` ficar próximo de `0,762` e `dec_com_den_iqs` próximo de `1,00`, a diferença veio do denominador.

Se ambos ficarem próximos de `1,00`, a diferença veio dos filtros do numerador: `ESTADO_INTRP = 4`, protocolo UCI, motivo nulo ou UC faturada.

Se o valor materializado no comparativo não bater com `dec_com_den_iqs`, então há divergência entre este notebook e o serviço `IndicadoresContinuidadeService`.